# S2 · Fundamentos de PySpark

**Integrante** · Joel Jonatan Quispe Mamani

**Dimensión** · Velocidad mediana de las aeronaves en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

Práctica sobre el histórico Bronze del 4 de enero de 2026 en UTC.

## 1. Lectura, transformaciones y plan

Abrimos un Parquet con esquema explícito, filtramos por segundos Unix y convertimos `time` a un timestamp UTC. La regla de elegibilidad es exploratoria.

In [2]:
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructType

bronze = Path("/opt/s01-arquitectura/data/2026")
bloque = bronze / "01" / "2026-01-01_2026-01-05.parquet"
dia_utc = date(2026, 1, 4)
inicio_utc = datetime.combine(dia_utc, time.min, tzinfo=timezone.utc)
fin_utc = inicio_utc + timedelta(days=1)

spark = (
    SparkSession.builder.master("local[2]")
    .appName("AeroStreamPeru-S02")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 15:14:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/24 15:14:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
esquema = (
    StructType()
    .add("time", LongType())
    .add("icao24", StringType())
    .add("lat", DoubleType())
    .add("lon", DoubleType())
    .add("onground", BooleanType())
    .add("velocity", DoubleType())
)
estados = spark.read.schema(esquema).parquet(str(bloque))
estados_del_dia = (
    estados.filter(
        (F.col("time") >= int(inicio_utc.timestamp()))
        & (F.col("time") < int(fin_utc.timestamp()))
    )
    .withColumn("captura_utc", F.timestamp_seconds("time"))
    .withColumn("icao24", F.lower(F.trim("icao24")))
)

identificador_valido = F.col("icao24").rlike(r"^[0-9a-f]{6}$")
posicion_en_zona = F.col("lat").between(-18.35, -0.04) & F.col("lon").between(-81.33, -68.65)
aeronave_en_vuelo = identificador_valido & posicion_en_zona & (~F.col("onground"))

estados_del_dia.printSchema()
estados_del_dia.explain(True)

root
 |-- time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- onground: boolean (nullable = true)
 |-- velocity: double (nullable = true)
 |-- captura_utc: timestamp (nullable = true)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(icao24, 'lower('trim('icao24)), None)]
+- Project [time#0L, icao24#1, lat#2, lon#3, onground#4, velocity#5, timestamp_seconds(time#0L) AS captura_utc#7]
   +- Filter ((time#0L >= cast(1767484800 as bigint)) AND (time#0L < cast(1767571200 as bigint)))
      +- Relation [time#0L,icao24#1,lat#2,lon#3,onground#4,velocity#5] parquet

== Analyzed Logical Plan ==
time: bigint, icao24: string, lat: double, lon: double, onground: boolean, velocity: double, captura_utc: timestamp
Project [time#0L, lower(trim(icao24#1, None)) AS icao24#8, lat#2, lon#3, onground#4, velocity#5, captura_utc#7]
+- Project [time#0L, icao24#1, lat#2, lon#3, onground#4, velocity#5, timesta

En `PushedFilters` aparecen los límites de `time`. Spark entrega el filtro temporal al lector Parquet. `filter` y `withColumn` construyen el plan, `explain` lo muestra y `show` ejecuta el trabajo para producir filas.

In [4]:
estados_del_dia.select("captura_utc", "icao24", "onground", "velocity").show(5)
print(f"Estados en el día de muestra: {estados_del_dia.count():,}")

+-------------------+------+--------+-----------------+
|        captura_utc|icao24|onground|         velocity|
+-------------------+------+--------+-----------------+
|2026-01-04 02:09:23|e80408|   false|             NULL|
|2026-01-04 02:09:24|e80408|   false|227.6657376748362|
|2026-01-04 02:09:25|e80408|   false|227.6657376748362|
|2026-01-04 02:09:26|e80408|   false|227.6657376748362|
|2026-01-04 02:09:27|e80408|   false|227.6657376748362|
+-------------------+------+--------+-----------------+
only showing top 5 rows


[Stage 1:=============================>                             (1 + 1) / 2]

Estados en el día de muestra: 579,911


## 2. Mi indicador por ventana de diez minutos

La población base son los estados elegibles en vuelo. Clasificamos las mediciones de velocidad en válidas, faltantes y fuera del rango exploratorio de 10 a 350 m/s, excluyendo los extremos. La mediana y los percentiles usan las válidas. La cobertura se expresa como la proporción de mediciones válidas dentro de la población base.

In [5]:
velocidad_en_rango = (F.col("velocity") > 10) & (F.col("velocity") < 350)
mediciones = estados_del_dia.filter(aeronave_en_vuelo).withColumn(
    "calidad_velocidad",
    F.when(F.col("velocity").isNull(), "faltante")
    .when(velocidad_en_rango, "valida")
    .otherwise("fuera_de_rango"),
)
velocidad_valida = F.when(F.col("calidad_velocidad") == "valida", F.col("velocity"))

resumen_velocidad = (
    mediciones.groupBy(F.window("captura_utc", "10 minutes").alias("ventana"))
    .agg(
        F.count("*").alias("observaciones_en_vuelo"),
        F.count(velocidad_valida).alias("velocidades_validas"),
        F.percentile_approx(velocidad_valida, [0.25, 0.50, 0.75]).alias("percentiles_m_s"),
    )
    .select(
        F.col("ventana.start").alias("inicio_utc"),
        "observaciones_en_vuelo",
        "velocidades_validas",
        F.round(100 * F.col("velocidades_validas") / F.col("observaciones_en_vuelo"), 2)
        .alias("porcentaje_valido"),
        F.round(F.get("percentiles_m_s", 0), 2).alias("velocidad_p25_t"),
        F.round(F.get("percentiles_m_s", 1), 2).alias("velocidad_mediana_t"),
        F.round(F.get("percentiles_m_s", 2), 2).alias("velocidad_p75_t"),
    )
    .orderBy("inicio_utc")
)
resumen_velocidad.show(8, truncate=False)

[Stage 4:=============================>                             (1 + 1) / 2]

+-------------------+----------------------+-------------------+-----------------+---------------+-------------------+---------------+
|inicio_utc         |observaciones_en_vuelo|velocidades_validas|porcentaje_valido|velocidad_p25_t|velocidad_mediana_t|velocidad_p75_t|
+-------------------+----------------------+-------------------+-----------------+---------------+-------------------+---------------+
|2026-01-04 00:00:00|5331                  |5331               |100.0            |232.65         |235.04             |237.49         |
|2026-01-04 00:10:00|5096                  |5096               |100.0            |232.53         |234.39             |237.62         |
|2026-01-04 00:20:00|3860                  |3860               |100.0            |231.41         |233.9              |236.6          |
|2026-01-04 00:30:00|4027                  |4027               |100.0            |203.11         |232.08             |239.42         |
|2026-01-04 00:40:00|5831                  |5831       

## 3. Ejercicio RDD

Comprobamos cuántas mediciones hay de cada categoría en la primera hora. La clasificación ya está definida en el DataFrame. El RDD reutiliza esa columna para que las reglas de calidad se mantengan en un solo lugar.

In [6]:
primera_hora = mediciones.filter(
    F.col("time") < int((inicio_utc + timedelta(hours=1)).timestamp())
)
conteo_por_calidad = (
    primera_hora.select("calidad_velocidad").rdd
    .map(lambda fila: (fila.calidad_velocidad, 1))
    .reduceByKey(lambda anterior, actual: anterior + actual)
)
for categoria, cantidad in conteo_por_calidad.takeOrdered(3, key=lambda par: par[0]):
    print(f"{categoria}: {cantidad} observaciones")

[Stage 7:=============================>                             (1 + 1) / 2]

valida: 31671 observaciones


## 4. Evidencia y defensa

Compara el total de observaciones en vuelo y el total de velocidades válidas. Explica por qué conviene mostrar esa cobertura junto a la mediana y qué información adicional aportan los percentiles 25 y 75.

Para la evidencia individual, ejecuta tu notebook y registra tu salida de la celda de agregación y el `PushedFilters` del plan. La guía pide que la captura muestre reloj y usuario o perfil.

In [7]:
spark.stop()